In [1]:
import pandas as pd
import numpy as np
import pickle
import torch

fname_base = 'D:\\PhD\\Openalex\\subset_ok\\years\\2019-2022\\'

In [ ]:
def load_node_csv(path, index_col, encoders=None, **kwargs):
    df = pd.read_csv(path, index_col=index_col, **kwargs)
    mapping = {index: i for i, index in enumerate(df.index.unique())}

    x = None
    if encoders is not None:
        xs = [encoder(df[col]) for col, encoder in encoders.items()]
        x = torch.cat(xs, dim=-1)

    return x, mapping

AUTHORS

ogni attributo è un tensore, poi cat

In [2]:
x_numeric = pd.read_csv(fname_base+'attributes\\authors_numeric.csv')
print(x_numeric.shape)

x_text = pd.read_csv(fname_base+'\\attributes\\authors_text_encoded.csv')
print(x_text.shape)

(15397, 6)
(15397, 2)


In [3]:
#x_df = pd.concat([x_numeric, x_text], axis=1)
x_df = x_numeric.merge(x_text, on='id')
print(x_df.shape)


(15397, 7)


In [5]:

x_df.to_csv(fname_base+'\\attributes\\authors_features.csv', index=False)
x_df.head(5)

,id,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded
0,A5009027337,282,17473,2.750000,41,74,"[0.022931719198822975, 0.003456080798059702, -..."
1,A5086198262,1312,238805,6.706977,153,620,"[-0.03792957216501236, 0.1179778128862381, 0.0..."
2,A5077097325,284,22719,0.000000,43,93,"[-0.03513007611036301, -0.012318928726017475, ..."
3,A5079098245,341,23334,3.545455,49,115,"[-0.003983520902693272, 0.055438704788684845, ..."
4,A5042195176,321,4841,0.184211,31,67,"[-0.02594980224967003, 0.0600975826382637, 0.0..."


In [12]:
#TEST

T1 = torch.Tensor([1, 2, 3, 4])
T2 = torch.Tensor([[0, 1, 2], [3,4,1], [1, 1, 0], [2, 0, 2]])

print("T1:", T1)
print(T1.shape)
print("T2:", T2)
print(T2.shape)

T1_new = T1.reshape(-1,1)
print("T1 new:", T1_new)
print(T1_new.shape)

T = torch.cat([T1_new,T2], axis=1)
print("T:",T)
print(T.shape)

T1: tensor([1., 2., 3., 4.])
torch.Size([4])
T2: tensor([[0., 1., 2.],
        [3., 4., 1.],
        [1., 1., 0.],
        [2., 0., 2.]])
torch.Size([4, 3])
T1 new: tensor([[1.],
        [2.],
        [3.],
        [4.]])
torch.Size([4, 1])
T: tensor([[1., 0., 1., 2.],
        [2., 3., 4., 1.],
        [3., 1., 1., 0.],
        [4., 2., 0., 2.]])
torch.Size([4, 4])


In [9]:
x_df = pd.read_csv(fname_base+'\\attributes\\authors_features.csv')
#print(type(x_df.values))
#tensor = torch.tensor(x_df.values)
#tensor.shape

#xs = [torch.tensor(x_df[c]) for c in x_df.columns]

x_df['name_encoded'] = x_df['name_encoded'].to_numpy()
x_df['name_encoded'].shape



(15397,)

In [13]:
def preprocessing_text_encoded(text):
    text = text[1:-1]
    list_str = text.split(', ')
    return [float(x) for x in list_str]

def preprocessing_text_encoded_abs(text):
    text = text[2:-1]
    #text = text.replace('\n','')
    delimiters = [" ", "\n"]
 
    for delimiter in delimiters:
        text = " ".join(text.split(delimiter))
    
    list_str = text.split()
    return [float(x) for x in list_str]

In [27]:
x_df = pd.read_csv(fname_base+'\\attributes\\authors_features.csv')
test1 = x_df['name_encoded'].tolist()[0][1:-1].split(', ')[0]
print(test1, type(test1))
x_df['name_encoded'] = x_df['name_encoded'].apply(preprocessing_text_encoded)
test2 = x_df['name_encoded'].tolist()[0][0]
print(test2, type(test2))

#tensor = torch.tensor(x_df.values)
#tensor.shape


0.022931719198822975 <class 'str'>
0.022931719198822975 <class 'float'>


In [28]:
x_df = pd.read_csv(fname_base+'\\attributes\\authors_features.csv')
x_df['name_encoded'] = x_df['name_encoded'].apply(preprocessing_text_encoded)
x_df.to_csv(fname_base+'\\attributes\\authors_features.csv', index=False)

In [30]:
x_df['name_encoded'] = x_df['name_encoded'].apply(lambda x: torch.tensor(x))


In [41]:
names = torch.stack(x_df['name_encoded'].tolist())
names.shape

torch.Size([15397, 384])

In [36]:
def mapping(df, col_index, col_id, fn):
    mapp = dict(zip(df[col_index], df[col_id]))
    with open(fname_base+fn+'.pkl', 'wb') as fp:
        pickle.dump(mapp, fp)
        print('dictionary saved successfully to file')

In [50]:
#we need a single dtaframe with also labels and id mapping
df_attr = pd.read_csv(fname_base+'\\attributes\\authors_features.csv')
print(df_attr.shape)
df_lab = pd.read_csv(fname_base+'A_labels.csv')
df_lab = df_lab.rename(columns={'author_id':'id'})
print(df_lab.shape)

A = df_attr.merge(df_lab, on='id')
print(A.shape)
A['aid'] = A.index
print(A.shape)

mapping_aid = mapping(A, 'aid', 'id', 'mapping_A')

A = A.drop('id', axis=1)
print(A.shape)

A.to_csv(fname_base+'\\attributes\\AX.csv', index=False)


(15397, 7)
(15397, 2)
(15397, 8)
(15397, 9)
dictionary saved successfully to file
(15397, 8)


In [64]:
AX = pd.read_csv(fname_base+'\\attributes\\AX.csv')
AX.dtypes

n_works                 int64
n_cit                   int64
impact_factor         float64
h_index                 int64
i10_index               int64
name_encoded           object
first_concept_name     object
aid                     int64
dtype: object

In [52]:
AX.head(5)

,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded,first_concept_name,aid
0,282,17473,2.750000,41,74,"[0.022931719198822975, 0.003456080798059702, -...",Economics and Business,0
1,1312,238805,6.706977,153,620,"[-0.03792957216501236, 0.1179778128862381, 0.0...",Philosophy and Art,1
2,284,22719,0.000000,43,93,"[-0.03513007611036301, -0.012318928726017475, ...",Mathematics,2
3,341,23334,3.545455,49,115,"[-0.003983520902693272, 0.055438704788684845, ...",Physics,3
4,321,4841,0.184211,31,67,"[-0.02594980224967003, 0.0600975826382637, 0.0...",Mathematics,4


In [65]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
AX['first_concept_name_enc'] = le.fit_transform(AX['first_concept_name'])
AX.head()

,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded,first_concept_name,aid,first_concept_name_enc
0,282,17473,2.750000,41,74,"[0.022931719198822975, 0.003456080798059702, -...",Economics and Business,0,0
1,1312,238805,6.706977,153,620,"[-0.03792957216501236, 0.1179778128862381, 0.0...",Philosophy and Art,1,4
2,284,22719,0.000000,43,93,"[-0.03513007611036301, -0.012318928726017475, ...",Mathematics,2,2
3,341,23334,3.545455,49,115,"[-0.003983520902693272, 0.055438704788684845, ...",Physics,3,5
4,321,4841,0.184211,31,67,"[-0.02594980224967003, 0.0600975826382637, 0.0...",Mathematics,4,2


In [61]:
data = AX[['first_concept_name', 'first_concept_name_enc']].drop_duplicates()
data
data_dict = dict(zip(data['first_concept_name_enc'], data['first_concept_name']))
with open(fname_base+'mapping_labels.pkl', 'wb') as fp:
        pickle.dump(data_dict, fp)
        print('dictionary saved successfully to file')
print(data_dict)

dictionary saved successfully to file
{0: 'Economics and Business', 4: 'Philosophy and Art', 2: 'Mathematics', 5: 'Physics', 3: 'Natural/Earth science and Medicine', 1: 'Engineering', 7: 'Psychology, Sociology and History', 6: 'Political science'}


In [66]:
AX = AX.drop('first_concept_name', axis=1)
AX = AX.rename(columns={'first_concept_name_enc': 'label'})
first_column = AX.pop('aid') 
AX.insert(0, 'aid', first_column) 
AX.head(5)


,aid,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded,label
0,0,282,17473,2.750000,41,74,"[0.022931719198822975, 0.003456080798059702, -...",0
1,1,1312,238805,6.706977,153,620,"[-0.03792957216501236, 0.1179778128862381, 0.0...",4
2,2,284,22719,0.000000,43,93,"[-0.03513007611036301, -0.012318928726017475, ...",2
3,3,341,23334,3.545455,49,115,"[-0.003983520902693272, 0.055438704788684845, ...",5
4,4,321,4841,0.184211,31,67,"[-0.02594980224967003, 0.0600975826382637, 0.0...",2


In [67]:
AX.to_csv(fname_base+'\\attributes\\AX.csv', index=False)

In [68]:
I_ohe = pd.read_csv(fname_base+'\\attributes\\institutions_ohe.csv')
I_ohe.columns

Index(['id', 'type_archive', 'type_company', 'type_education', 'type_facility',
       'type_government', 'type_healthcare', 'type_nonprofit', 'type_other',
       'country-code_AE',
       ...
       'country-code_TR', 'country-code_TW', 'country-code_UA',
       'country-code_UG', 'country-code_US', 'country-code_UY',
       'country-code_UZ', 'country-code_VN', 'country-code_XK',
       'country-code_ZA'],
      dtype='object', length=106)

In [74]:
I_text = pd.read_csv(fname_base+'\\attributes\\institutions_text_encoded.csv')
I_text = I_text.drop('name', axis=1)
I_text['name_encoded'] = I_text['name_encoded'].apply(preprocessing_text_encoded)
I = I_ohe.merge(I_text, on='id')
I.shape
I.to_csv(fname_base+'\\attributes\\institutions_features.csv', index=False)

In [29]:
#subset ultimo (si spera)

authors = pd.read_csv(fname_base+'attributes\\authors_features.csv')['id'].tolist()
print('No. authors', len(authors))

papers = pd.read_csv(fname_base+'attributes\\paper_concepts_lab_ohe.csv')['paper_id'].tolist()
print('No. papers',len(papers))

PA_old = pd.read_csv(fname_base+'PA.csv')
print('PA old', PA_old.shape)
PA = PA_old[(PA_old['author'].isin(authors)) & (PA_old['paper'].isin(papers))]
print('PA new', PA.shape)
PA.to_csv(fname_base+'PA.csv', index=False)

#gli authors sono sempre gli stessi o ne abbiamo perso qualcuno? ok tutto a posto
print('No authors who wrote at least one paper', len(PA['author'].drop_duplicates().tolist()))

PP_old = pd.read_csv(fname_base+'PP.csv')
print('PP old', PP_old.shape)
PP = PP_old[(PP_old['citing'].isin(papers)) & (PP_old['cited'].isin(papers))]
print('PP new', PP.shape)
PP.to_csv(fname_base+'PP.csv', index=False)

AI_old = pd.read_csv(fname_base+'AI.csv')
print('AI old', AI_old.shape)
AI = AI_old[AI_old['author_id'].isin(authors)]
print('AI new', AI.shape)
AI.to_csv(fname_base+'AI.csv', index=False)

institutions = AI['institution_id'].drop_duplicates().tolist()
print('No institutions', len(institutions))





No. authors 15397
No. papers 10173
PA old (21457, 3)
PA new (18768, 3)
No authors who wrote at least one paper 15397
PP old (272, 3)
PP new (192, 3)
AI old (4896, 2)
AI new (4893, 2)
No institutions 1830


In [33]:
I = pd.read_csv(fname_base+'attributes\\institutions_features.csv')
print(I.shape)
I = I[I['id'].isin(institutions)]
print(I.shape)
I.to_csv(fname_base+'attributes\\institutions_features.csv', index=False)

(1831, 107)
(1830, 107)


In [37]:
I = pd.read_csv(fname_base+'attributes\\institutions_features.csv')
print(I.shape)

I['iid'] = I.index
print(I.shape)

mapping_iid = mapping(I, 'iid', 'id', 'mapping_I')

I = I.drop('id', axis=1)
print(I.shape)

I.to_csv(fname_base+'\\attributes\\IX.csv', index=False)

(1830, 107)
(1830, 108)
dictionary saved successfully to file
(1830, 107)


PAPERS 

In [12]:
P_numeric = pd.read_csv(fname_base+'attributes\\papers_numeric.csv') #id, publication_year, num_citations
print(P_numeric.shape)
P_numeric = P_numeric.drop_duplicates(subset='id')
print(P_numeric.shape)


P_ohe = pd.read_csv(fname_base+'attributes\\paper_concepts_lab_ohe.csv')
print(P_ohe.columns)
print(P_ohe.shape)

P_text = pd.read_csv(fname_base+'attributes\\papers_text_encoded.csv')
print(P_text.columns)
print(P_text.shape)

P_abs = pd.read_csv(fname_base+'attributes\\paper_abstracts_encoded.csv')
print(P_abs.columns)
print(P_abs.shape)
data = P_abs['id'].tolist()[0]
print(type(data))

(10174, 3)
(10174, 3)
0        False
1        False
2        False
3        False
4        False
         ...  
10169    False
10170    False
10171    False
10172    False
10173    False
Length: 10174, dtype: bool
(10173, 3)
Index(['paper_id', 'Computer science', 'Economics and Business', 'Engineering',
       'Mathematics', 'Natural/Earth science and Medicine',
       'Philosophy and Art', 'Physics', 'Political science',
       'Psychology, Sociology and History'],
      dtype='object')
(10173, 10)
Index(['id', 'title', 'title_encoded'], dtype='object')
(10173, 3)
Index(['id', 'abstract_encoded'], dtype='object')
(10173, 2)
<class 'str'>


In [18]:
papers = pd.read_csv(fname_base+'attributes\\paper_concepts_lab_ohe.csv')['paper_id'].tolist()
print('papers with attributes',len(papers))

papers_all = PA['paper'].drop_duplicates().tolist()
print('papers in PA', len(papers_all))

papers_ok = PA[PA['paper'].isin(papers)]['paper'].drop_duplicates().tolist()
print('papers in PA with attributes', len(papers_ok))

papers_to_check = PA[~PA['paper'].isin(papers)]['paper'].drop_duplicates().tolist()
print('papers in PA to check', len(papers_to_check))

print('papers in PA', len(papers_ok)+len(papers_to_check))




papers with attributes 10173
papers in PA 12128
papers in PA with attributes 10150
papers in PA to check 1978
papers in PA 12128


In [23]:
#vediamo i papers in PA to check di che hanno sono

P = pd.read_csv('D:\\PhD\\Openalex\\papers.csv')[['id', 'publication_year']]

P_check = P[P['id'].isin(papers_to_check)]
print(P_check.shape)

P_sub = P_check[(P_check['publication_year'] >= 2019) & (P_check['publication_year'] <= 2022)]
print(P_sub.shape)

# OK SONO TUTTI PAPER DI ALTRI ANNI (PRECEDENTI O SUCCESSIVI)

(1978, 2)
(0, 2)


In [63]:
P_numeric = pd.read_csv(fname_base+'attributes\\papers_numeric.csv') #id, publication_year, num_citations
print(P_numeric.shape)
P_numeric = P_numeric.drop_duplicates(subset='id')
print(P_numeric.shape)

P_ohe = pd.read_csv(fname_base+'attributes\\paper_concepts_lab_ohe.csv')
P_ohe = P_ohe.rename(columns={'paper_id':'id'})
print(P_ohe.shape)

P_text = pd.read_csv(fname_base+'attributes\\papers_text_encoded.csv')
P_text = P_text.drop('title', axis=1)
P_text['title_encoded'] = P_text['title_encoded'].apply(preprocessing_text_encoded)
print(P_text.shape)

P_abs = pd.read_csv(fname_base+'attributes\\paper_abstracts_encoded.csv')
#P_abs['abstract_encoded'] = P_abs['abstract_encoded'].apply(lambda x: x.replace('\n',''))
P_abs['abstract_encoded'] = P_abs['abstract_encoded'].apply(preprocessing_text_encoded_abs)
print(P_abs.shape)

P = P_numeric.merge(P_ohe, on='id')
P = P.merge(P_text, on='id')
P = P.merge(P_abs, on='id')
print(P.shape)

P.to_csv(fname_base+'attributes\\papers_features.csv', index=False)

P['pid'] = P.index
print(P.shape)

mapping_pid = mapping(P, 'pid', 'id', 'mapping_P')

P = P.drop('id', axis=1)
print(P.shape)

P.to_csv(fname_base+'\\attributes\\PX.csv', index=False)


(10174, 3)
(10173, 3)
(10173, 10)
(10173, 2)
(10173, 2)
(10173, 14)
(10173, 15)
dictionary saved successfully to file
(10173, 14)


In [11]:
#mapping edges

def get_mapping(pckl):
    file = open(pckl, 'rb')
    diz = pickle.load(file)
    reversedDict = {val: key for (key, val) in diz.items()}
    return reversedDict
    


In [13]:
from sklearn.preprocessing import OrdinalEncoder



mapping_A = get_mapping(fname_base+'mapping_A.pkl')
mapping_P = get_mapping(fname_base+'mapping_P.pkl')
mapping_I = get_mapping(fname_base+'mapping_I.pkl')

mapping_positions = {'last':0, 'middle':1, 'first':2}

PA = pd.read_csv(fname_base+'PA.csv') #paper', 'author', 'position
#PA['position'] = OrdinalEncoder().fit_transform(PA['position'])
PA.replace({'paper': mapping_P}, inplace=True)
PA.replace({'author': mapping_A}, inplace=True)
PA.replace({'position': mapping_positions}, inplace=True)
PA.to_csv(fname_base+'PA_map.csv', index=False)
print(PA.head(3))

PP = pd.read_csv(fname_base+'PP.csv') #'citing', 'cited', 'year'
PP.replace({'citing': mapping_P}, inplace=True)
PP.replace({'cited': mapping_P}, inplace=True)
PP.to_csv(fname_base+'PP_map.csv', index=False)
print(PP.head(3))

AI = pd.read_csv(fname_base+'AI.csv') #'author_id', 'institution_id'
AI.replace({'author_id': mapping_A}, inplace=True)
AI.replace({'institution_id': mapping_I}, inplace=True)
AI.to_csv(fname_base+'AI_map.csv', index=False)
print(AI.head(3))



   paper  author  position
0   2968       0         2
1      0       2         2
2      1       6         2
   citing  cited  year
0    2982   2968  2020
1    2985   2985  2020
2      36     79  2019
   author_id  institution_id
0          1              59
1          2              88
2          4             175


FEATURES

In [6]:
PX = pd.read_csv(fname_base+'attributes\\PX.csv')
first_column = PX.pop('pid') 
PX.insert(0, 'pid', first_column)
PX.to_csv(fname_base+'attributes\\PX.csv', index=False)

IX = pd.read_csv(fname_base+'attributes\\IX.csv')
first_column = IX.pop('iid') 
IX.insert(0, 'iid', first_column)
IX.to_csv(fname_base+'attributes\\IX.csv', index=False)

In [9]:
#feature scaling
from sklearn.preprocessing import StandardScaler

def feature_scaling(X, cols_numeric):
    scaler = StandardScaler()
    X[cols_numeric] = scaler.fit_transform(X[cols_numeric])
    return X
 

In [10]:
# authors
AX = pd.read_csv(fname_base+'attributes\\AX.csv')
cols_numeric_A = ['n_works', 'n_cit', 'impact_factor', 'h_index', 'i10_index']
AX = feature_scaling(AX, cols_numeric_A)
AX.to_csv(fname_base+'attributes\\AX.csv', index=False)

# papers
PX = pd.read_csv(fname_base+'attributes\\PX.csv')
cols_numeric_P = ['num_citations'] #publication_year
PX = feature_scaling(PX, cols_numeric_P)
PX.to_csv(fname_base+'attributes\\PX.csv', index=False)


In [2]:
#authors

AX = pd.read_csv(fname_base+'attributes\\AX.csv')
AX.head(3)

#x_df['name_encoded'] = x_df['name_encoded'].apply(lambda x: torch.tensor(x))
#names = torch.stack(x_df['name_encoded'].tolist())
#names.shape

,aid,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded,label
0,0,282,17473,2.750000,41,74,"[0.022931719198822975, 0.003456080798059702, -...",0
1,1,1312,238805,6.706977,153,620,"[-0.03792957216501236, 0.1179778128862381, 0.0...",4
2,2,284,22719,0.000000,43,93,"[-0.03513007611036301, -0.012318928726017475, ...",2


In [11]:
#authors

AX = pd.read_csv(fname_base+'attributes\\AX.csv')
AX.head(3)

,aid,n_works,n_cit,impact_factor,h_index,i10_index,name_encoded,label
0,0,0.662556,2.282264,0.498463,1.761914,0.564564,"[0.022931719198822975, 0.003456080798059702, -...",0
1,1,4.770417,34.678924,1.965692,8.726855,7.448775,"[-0.03792957216501236, 0.1179778128862381, 0.0...",4
2,2,0.670532,3.050128,-0.521224,1.886288,0.804124,"[-0.03513007611036301, -0.012318928726017475, ...",2


In [58]:
#authors

AX = pd.read_csv(fname_base+'attributes\\AX.csv')

col_text = 'name_encoded'
AX[col_text] = AX[col_text].apply(preprocessing_text_encoded)
AX[col_text] = AX[col_text].apply(lambda x: torch.tensor(x))
col_text_torch = torch.stack(AX['name_encoded'].tolist())
cols_numeric = [c for c in AX.columns.tolist()[1:-1] if c != col_text ]
cols_numeric_torch = [torch.tensor(AX[c].tolist()) for c in cols_numeric]

X = torch.cat([col_text_torch, cols_numeric_torch[0].unsqueeze(dim=1)], dim=1)
for x in cols_numeric_torch[1:]:
    X = torch.cat([X, x.unsqueeze(dim=1)], dim=1)

print(X.shape)
torch.save(X, fname_base+'attributes\\AX_tensor.pt')

col_label = 'label'
Y = torch.tensor(AX[col_label].tolist())
print(Y.shape)
torch.save(Y, fname_base+'attributes\\AY_tensor.pt')


torch.Size([15397, 389])
torch.Size([15397])


In [46]:
data = torch.load(fname_base+'AX_tensor.pt')
data.shape

torch.Size([15397, 390])

In [22]:
n_works = torch.tensor(AX['n_works'].tolist())
print(n_works.shape)

n_cit = torch.tensor(AX['n_cit'].tolist())
print(n_cit.shape)

torch.Size([15397])
torch.Size([15397])


In [36]:
x = torch.empty(size=(3,2))
print(x)
print(x.shape)
y = torch.tensor([ 0.3981,  0.6952, -1.2320])
print(y.shape)

y1 = y.unsqueeze(dim=1) # same as y but with shape (1,3)
print(y1.shape)
x = torch.cat([x,y1], dim=1)
print(x)

tensor([[0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00],
        [0.0000e+00, 6.8664e-44]])
torch.Size([3, 2])
torch.Size([3])
torch.Size([3, 1])
tensor([[ 0.0000e+00,  0.0000e+00,  3.9810e-01],
        [ 0.0000e+00,  0.0000e+00,  6.9520e-01],
        [ 0.0000e+00,  6.8664e-44, -1.2320e+00]])


In [7]:
PX = pd.read_csv(fname_base+'attributes\\PX.csv')
PX.head(3)

,pid,publication_year,num_citations,Computer science,Economics and Business,Engineering,Mathematics,Natural/Earth science and Medicine,Philosophy and Art,Physics,Political science,"Psychology, Sociology and History",title_encoded,abstract_encoded
0,0,2019,1786,1,1,1,0,0,1,0,0,1,"[-0.0038463666569441557, -0.006265648175030947...","[0.072352618, 0.0581676811, -0.0435537137, 0.0..."
1,1,2019,672,1,0,0,0,1,0,0,0,1,"[-0.024850422516465187, -0.09042566269636154, ...","[0.00240442739, -0.0407005735, -0.0767312124, ..."
2,2,2019,492,1,0,0,1,0,0,0,0,0,"[0.01041603647172451, -0.005232919938862324, 0...","[0.00240442739, -0.0407005735, -0.0767312124, ..."


In [59]:
#papers

PX = pd.read_csv(fname_base+'attributes\\PX.csv')

col_text = 'title_encoded'
PX[col_text] = PX[col_text].apply(preprocessing_text_encoded)
PX[col_text] = PX[col_text].apply(lambda x: torch.tensor(x))
col_text_torch = torch.stack(PX[col_text].tolist())

col_text_abs = 'abstract_encoded'
PX[col_text_abs] = PX[col_text_abs].apply(preprocessing_text_encoded)
PX[col_text_abs] = PX[col_text_abs].apply(lambda x: torch.tensor(x))
col_text_abs_torch = torch.stack(PX[col_text_abs].tolist())

cols_numeric = [c for c in PX.columns.tolist()[1:] if (c != col_text and c!= col_text_abs)]
cols_numeric_torch = [torch.tensor(PX[c].tolist()) for c in cols_numeric]

X = torch.cat([col_text_torch, col_text_abs_torch], dim=1)
print(X.shape)

for x in cols_numeric_torch:
    X = torch.cat([X, x.unsqueeze(dim=1)], dim=1)

print(X.shape)
torch.save(X, fname_base+'attributes\\PX_tensor.pt')


torch.Size([10173, 768])
torch.Size([10173, 779])


In [8]:
IX = pd.read_csv(fname_base+'attributes\\IX.csv')
IX.head(3)

,iid,type_archive,type_company,type_education,type_facility,type_government,type_healthcare,type_nonprofit,type_other,country-code_AE,...,country-code_TW,country-code_UA,country-code_UG,country-code_US,country-code_UY,country-code_UZ,country-code_VN,country-code_XK,country-code_ZA,name_encoded
0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,"[0.031768061220645905, -0.03628000244498253, 0..."
1,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,"[-0.07017671316862106, 0.02301018126308918, -0..."
2,2,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"[-0.03410424664616585, 0.03702332824468613, -0..."


In [60]:
#institutions

IX = pd.read_csv(fname_base+'attributes\\IX.csv')

col_text = 'name_encoded'
IX[col_text] = IX[col_text].apply(preprocessing_text_encoded)
IX[col_text] = IX[col_text].apply(lambda x: torch.tensor(x))
col_text_torch = torch.stack(IX[col_text].tolist())
cols_numeric = [c for c in IX.columns.tolist()[1:] if c != col_text ]
cols_numeric_torch = [torch.tensor(IX[c].tolist()) for c in cols_numeric]

X = torch.cat([col_text_torch, cols_numeric_torch[0].unsqueeze(dim=1)], dim=1)
for x in cols_numeric_torch[1:]:
    X = torch.cat([X, x.unsqueeze(dim=1)], dim=1)

print(X.shape)
torch.save(X, fname_base+'attributes\\IX_tensor.pt')

torch.Size([1830, 489])


In [62]:
from scipy.sparse import coo_matrix

def edge_index_generation(df):
    return coo_matrix(df.values.T)

In [68]:
AI = pd.read_csv(fname_base+'AI_map.csv')
print(AI.shape)
AI_coo = edge_index_generation(AI)
print(AI_coo.shape)

(4893, 2)
(2, 4893)


In [69]:
AI.columns

Index(['author_id', 'institution_id'], dtype='object')